# Vendor Performance Analysis

This notebook has been refactored to make the analysis easier to maintain, faster to run, and clearer to explain to business users. The workflow now separates: 

1. **Data loading and reusable helper logic**.
2. **Codebook and metric definitions**.
3. **Analysis questions with documented engineering logic and business logic**.
4. **Insights written directly below each question**.

> **Important note:** the numeric insights written in markdown below are based on the last successful notebook execution that was already saved in this file. The repository currently does not contain a populated `inventory.db`, so the notebook code is prepared for rerun when the data is available again.

## Business Questions Covered

This notebook answers the following questions:

1. Which vendors and brands drive the highest sales revenue?
2. Which vendors contribute the most to total purchase spend?
3. How concentrated is procurement among top vendors?
4. How many vendors drive roughly 80% of purchase spend?
5. Which brands have low sales but high margins?
6. What is the relationship between purchase quantity and unit cost?
7. What purchase quantity band appears optimal for cost savings?
8. Which vendors have low inventory turnover?
9. How much capital is locked in unsold inventory by vendor?
10. Which vendors contribute the most to locked working capital?
11. How does profit margin distribution differ between high- and low-performing vendors?
12. What are the 95% confidence intervals for profit margin by performance group?

In [ ]:
import sqlite3
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter, PercentFormatter
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DB_PATH = "inventory.db"
SUMMARY_TABLE = "vendor_sales_summary"
BASE_FILTER = "GrossProfit > 0 AND ProfitMargin > 0 AND TotalSalesQuantity > 0"


def get_connection(db_path: str = DB_PATH) -> sqlite3.Connection:
    """Create a SQLite connection for repeatable notebook queries."""
    return sqlite3.connect(db_path)


def table_exists(conn: sqlite3.Connection, table_name: str) -> bool:
    """Return True when the target table exists in SQLite."""
    query = "SELECT name FROM sqlite_master WHERE type = 'table' AND name = ?"
    return conn.execute(query, (table_name,)).fetchone() is not None


def read_sql(query: str, conn: sqlite3.Connection, params: tuple | None = None) -> pd.DataFrame:
    """Wrapper around pandas SQL reads for cleaner notebook code."""
    return pd.read_sql_query(query, conn, params=params)


def fmt_millions(x, _=None):
    return f"{x / 1_000_000:.1f}M"


def add_bar_labels(ax, axis="x", scale=1_000_000, suffix="M"):
    """Add simple labels to matplotlib/seaborn bar charts."""
    for patch in ax.patches:
        if axis == "x":
            width = patch.get_width()
            y = patch.get_y() + patch.get_height() / 2
            ax.text(width, y, f" {width / scale:.1f}{suffix}", va="center", ha="left", fontsize=9)
        else:
            height = patch.get_height()
            x = patch.get_x() + patch.get_width() / 2
            ax.text(x, height, f"{height / scale:.1f}{suffix}", va="bottom", ha="center", fontsize=9)


def mean_ci(series: pd.Series, confidence: float = 0.95) -> tuple[float, float, float]:
    """Return mean and confidence interval for a numeric pandas series."""
    series = series.dropna()
    n = len(series)
    mean = series.mean()
    if n < 2:
        return mean, mean, mean
    se = stats.sem(series)
    half_width = se * stats.t.ppf((1 + confidence) / 2, n - 1)
    return mean, mean - half_width, mean + half_width


conn = get_connection()
if table_exists(conn, SUMMARY_TABLE):
    summary_df = read_sql(f"SELECT * FROM {SUMMARY_TABLE}", conn)
    analysis_df = read_sql(
        f"SELECT * FROM {SUMMARY_TABLE} WHERE {BASE_FILTER}",
        conn,
    )
    print(f"Summary rows loaded: {len(summary_df):,}")
    print(f"Analysis-ready rows after quality filter: {len(analysis_df):,}")
    display(summary_df.head())
else:
    summary_df = pd.DataFrame()
    analysis_df = pd.DataFrame()
    print(
        "`vendor_sales_summary` was not found in inventory.db. "
        "Load the database first, then rerun this notebook to refresh the analysis."
    )

## Codebook

The curated table used in this notebook is `vendor_sales_summary`. It is produced by joining and aggregating raw purchase, sales, price, and freight data. The codebook below documents the most important fields used throughout the analysis.

### Column Definitions

| Column | Meaning | Analytical use |
|---|---|---|
| `VendorNumber` | Unique vendor identifier. | Grouping, ranking, risk concentration. |
| `VendorName` | Vendor name cleaned at transformation time. | Human-readable supplier reporting. |
| `Brand` | Brand or product-category-like identifier. | Brand-level performance analysis. |
| `PurchasePrice` | Average recorded purchase unit price from `purchases`. | Cost analysis, bulk-buy evaluation. |
| `ActualPrice` | Average reference price from `purchase_prices`. | Benchmarking purchase efficiency. |
| `TotalPurchaseQuantity` | Total purchased units by vendor-brand. | Turnover, demand-supply balance. |
| `TotalPurchaseDollars` | Total purchase spend by vendor-brand. | Procurement concentration and spend analysis. |
| `TotalSalesQuantity` | Total sold units by vendor-brand. | Turnover and sell-through analysis. |
| `TotalSalesDollars` | Total sales revenue by vendor-brand. | Revenue ranking and profitability. |
| `TotalSalesPrice` | Sum of sales price values. | Supplemental commercial analysis. |
| `TotalExciseTax` | Total excise tax captured in sales data. | Net economics context. |
| `FreightCost` | Vendor-level freight cost from invoices. | Landed-cost interpretation. |
| `GrossProfit` | `TotalSalesDollars - TotalPurchaseDollars`. | Core profitability metric. |
| `ProfitMargin` | `GrossProfit / TotalSalesDollars * 100`. | Margin segmentation. |
| `StockTurnover` | `TotalSalesQuantity / TotalPurchaseQuantity`. | Slow-moving inventory detection. |
| `SalesToPurchaseRatio` | `TotalSalesDollars / TotalPurchaseDollars`. | Commercial efficiency check. |

### Metric Caveat

Because the repository currently does not include a populated database snapshot, treat the written insights in this notebook as tied to the **last saved execution state**. When fresh data is loaded into `inventory.db`, rerun the notebook to refresh all tables, charts, and narrative.

In [ ]:
codebook_df = pd.DataFrame(
    [
        ("VendorNumber", "Vendor identifier", "Primary entity key for grouping vendors"),
        ("VendorName", "Vendor display name", "Readable supplier label for reporting"),
        ("Brand", "Brand/product grouping", "Brand-level revenue and margin analysis"),
        ("PurchasePrice", "Average purchase unit price", "Cost curve and bulk-buy analysis"),
        ("ActualPrice", "Average reference price", "Benchmark against purchase price"),
        ("TotalPurchaseQuantity", "Total purchased units", "Turnover denominator and stock exposure"),
        ("TotalPurchaseDollars", "Total purchase spend", "Spend concentration and procurement analytics"),
        ("TotalSalesQuantity", "Total sold units", "Sell-through and turnover analytics"),
        ("TotalSalesDollars", "Total realized revenue", "Sales ranking and commercial performance"),
        ("TotalSalesPrice", "Total sales price", "Supplementary commercial metric"),
        ("TotalExciseTax", "Total excise tax", "Tax-adjusted context"),
        ("FreightCost", "Allocated freight cost", "Landed cost interpretation"),
        ("GrossProfit", "Sales minus purchase dollars", "Absolute profitability"),
        ("ProfitMargin", "Gross profit as % of sales", "Margin screening"),
        ("StockTurnover", "Sales quantity / purchase quantity", "Slow-moving inventory risk"),
        ("SalesToPurchaseRatio", "Sales dollars / purchase dollars", "Commercial efficiency ratio"),
    ],
    columns=["column", "definition", "used_for"],
)
codebook_df

## What Was Done in This Notebook

### Documentation of the Work Performed

This notebook was designed to analyze vendor performance from a curated vendor-brand summary table. The work can be described in three layers:

#### 1. Data preparation work
- Loaded the curated `vendor_sales_summary` table from SQLite instead of repeatedly joining raw tables in each analysis step.
- Applied a reusable analysis filter to remove clearly inconsistent records for the performance-focused sections: positive `GrossProfit`, positive `ProfitMargin`, and positive `TotalSalesQuantity`.
- Standardized plotting and formatting functions so every chart uses the same style and readable units.

#### 2. Data engineering logic applied
- **Push heavy aggregation to SQL first** so grouped spend, sales, turnover, and inventory logic are computed closer to the database.
- **Use one curated summary table** (`vendor_sales_summary`) as the semantic layer for analysis. This reduces notebook complexity and avoids duplicated join logic.
- **Use reusable helper functions** for database reads, chart labeling, and confidence interval calculations to reduce repeated code.
- **Use quantile-based thresholds** for identifying low-sales/high-margin brands, low-turnover vendors, and purchase quantity bands. This is more robust than hard-coded cutoffs because it adapts to the data distribution.
- **Use window-style cumulative logic** for Pareto and supplier concentration analysis.
- **Use unit-cost approximation for locked capital** by multiplying estimated unsold quantity by average purchase cost.

#### 3. Business logic applied
- **Revenue leadership** is interpreted using `TotalSalesDollars`.
- **Procurement dependency risk** is interpreted using concentration of `TotalPurchaseDollars` among top vendors.
- **Promotion/pricing opportunity** is defined as brands with **low sales but high margin**, because those brands may have room for better activation without immediately destroying profitability.
- **Bulk-buy discount effect** is studied through the relationship between purchase `Quantity` and `PurchasePrice`.
- **Optimal quantity band** is defined as the stable purchase-size bucket with the lowest average unit cost.
- **Slow-moving inventory risk** is identified by low `StockTurnover` or low vendor-level turnover.
- **Locked working capital** is estimated from unsold quantity multiplied by average unit cost, highlighting money tied up in stock.
- **Vendor performance groups** are defined using sales quartiles, then compared using profit margin distributions and 95% confidence intervals.

### Why the Notebook Was Optimized

The original notebook answered the right questions, but it repeated logic in several places and mixed business explanation with ad-hoc code. The refactored version improves that by:

- reducing duplicate plotting code,
- centralizing SQL/data access logic,
- documenting assumptions in markdown, and
- writing business insights immediately after each analytical question.

In [ ]:
quality_snapshot = pd.DataFrame(
    {
        "dataset": ["summary_df", "analysis_df"],
        "rows": [len(summary_df), len(analysis_df)],
        "vendors": [summary_df["VendorNumber"].nunique(), analysis_df["VendorNumber"].nunique()],
        "brands": [summary_df["Brand"].nunique(), analysis_df["Brand"].nunique()],
        "total_sales_dollars": [summary_df["TotalSalesDollars"].sum(), analysis_df["TotalSalesDollars"].sum()],
        "total_purchase_dollars": [summary_df["TotalPurchaseDollars"].sum(), analysis_df["TotalPurchaseDollars"].sum()],
    }
)
quality_snapshot

## Question 1 — Which vendors and brands drive the highest sales revenue?

In [ ]:
top_vendor_sales = (
    analysis_df.groupby("VendorName", as_index=False)["TotalSalesDollars"]
    .sum()
    .sort_values("TotalSalesDollars", ascending=False)
    .head(10)
)

top_brand_sales = (
    analysis_df.groupby("Brand", as_index=False)["TotalSalesDollars"]
    .sum()
    .sort_values("TotalSalesDollars", ascending=False)
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(data=top_vendor_sales, y="VendorName", x="TotalSalesDollars", ax=axes[0], color="teal")
axes[0].set_title("Top Vendors by Sales Revenue")
axes[0].set_xlabel("Total Sales Dollars")
axes[0].xaxis.set_major_formatter(FuncFormatter(fmt_millions))
add_bar_labels(axes[0], axis="x")

sns.barplot(data=top_brand_sales.assign(Brand=top_brand_sales["Brand"].astype(str)), x="Brand", y="TotalSalesDollars", ax=axes[1], color="coral")
axes[1].set_title("Top Brands by Sales Revenue")
axes[1].set_xlabel("Brand")
axes[1].set_ylabel("Total Sales Dollars")
axes[1].yaxis.set_major_formatter(FuncFormatter(fmt_millions))
axes[1].tick_params(axis="x", rotation=45)
add_bar_labels(axes[1], axis="y")

plt.tight_layout()
plt.show()

(top_vendor_sales, top_brand_sales)

### Insights

- The last saved execution shows **DIAGEO NORTH AMERICA INC** as the strongest revenue-driving vendor at **$67.99M**, followed by **MARTIGNETTI COMPANIES** at **$39.33M** and **PERNOD RICARD USA** at **$32.06M**.
- At the brand level, the top three sales-driving brands were **1233 ($5.10M)**, **3405 ($4.82M)**, and **8068 ($4.54M)**.
- This means revenue is being driven by a relatively small set of high-volume vendors and a narrower set of high-performing brands.
- A practical business follow-up is to protect availability, trade support, and forecasting quality for these top revenue contributors first.

## Question 2 — Which vendors contribute the most to total purchase spend?

## Question 3 — How concentrated is procurement among top vendors?

## Question 4 — How many vendors drive roughly 80% of purchase spend?

In [ ]:
vendor_purchase = read_sql(
    f"""
    SELECT
        VendorNumber,
        VendorName,
        SUM(TotalPurchaseDollars) AS total_purchase_dollars
    FROM {SUMMARY_TABLE}
    GROUP BY VendorNumber, VendorName
    HAVING SUM(TotalPurchaseDollars) > 0
    ORDER BY total_purchase_dollars DESC
    """,
    conn,
)

vendor_purchase["purchase_share_pct"] = (
    vendor_purchase["total_purchase_dollars"] / vendor_purchase["total_purchase_dollars"].sum() * 100
)
vendor_purchase["cumulative_pct"] = vendor_purchase["purchase_share_pct"].cumsum()

fig, ax1 = plt.subplots(figsize=(14, 6))
pareto_plot = vendor_purchase.head(20).copy()
ax1.bar(pareto_plot["VendorName"], pareto_plot["total_purchase_dollars"], color="steelblue")
ax1.set_title("Pareto Chart: Vendor Contribution to Purchase Spend")
ax1.set_xlabel("Vendor")
ax1.set_ylabel("Purchase Dollars")
ax1.yaxis.set_major_formatter(FuncFormatter(fmt_millions))
ax1.tick_params(axis="x", rotation=60)

ax2 = ax1.twinx()
ax2.plot(pareto_plot["VendorName"], pareto_plot["cumulative_pct"], color="crimson", marker="o")
ax2.axhline(80, color="darkgreen", linestyle="--", linewidth=1.5)
ax2.set_ylabel("Cumulative Contribution (%)")
ax2.yaxis.set_major_formatter(PercentFormatter())
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()

vendors_for_80 = int((vendor_purchase["cumulative_pct"] <= 80).sum() + 1)
concentration_summary = pd.DataFrame(
    {
        "Metric": [
            "Total Procurement",
            "Dependency on Top 1 Vendor",
            "Dependency on Top 3 Vendors",
            "Dependency on Top 5 Vendors",
            "Dependency on Top 10 Vendors",
            "Vendors Needed to Reach ~80%",
        ],
        "Value": [
            f"{vendor_purchase['total_purchase_dollars'].sum() / 1_000_000:.2f}M",
            f"{vendor_purchase.head(1)['purchase_share_pct'].sum():.2f}%",
            f"{vendor_purchase.head(3)['purchase_share_pct'].sum():.2f}%",
            f"{vendor_purchase.head(5)['purchase_share_pct'].sum():.2f}%",
            f"{vendor_purchase.head(10)['purchase_share_pct'].sum():.2f}%",
            f"{vendors_for_80} vendors ({vendor_purchase.head(vendors_for_80)['purchase_share_pct'].sum():.2f}%)",
        ],
    }
)

(vendor_purchase.head(10), concentration_summary)

### Insights

- The largest procurement-spend vendor in the saved output is **DIAGEO NORTH AMERICA INC** at **$50.96M**, followed by **MARTIGNETTI COMPANIES ($27.85M)**, **JIM BEAM BRANDS COMPANY ($24.20M)**, and **PERNOD RICARD USA ($24.12M)**.
- Total procurement spend in the saved analysis equals **$320.91M**.
- Supplier dependency appears **meaningful but not extreme**: the **top 1 vendor** accounts for **15.88%** of spend, the **top 3** for **32.10%**, the **top 5** for **45.11%**, and the **top 10** for **65.49%**.
- The Pareto view shows that roughly **17 vendors account for 80.22%** of total purchase dollars.
- Business interpretation: procurement is concentrated enough to justify vendor-risk monitoring, but not so concentrated that the company is dependent on only a handful of suppliers.

## Question 5 — Which brands have low sales but high margins?

In [ ]:
brand_perf = read_sql(
    f"""
    SELECT
        Brand,
        VendorName,
        SUM(TotalSalesDollars) AS TotalSalesDollars,
        AVG(ProfitMargin) AS ProfitMargin,
        SUM(GrossProfit) AS GrossProfit,
        SUM(TotalSalesQuantity) AS TotalSalesQuantity
    FROM {SUMMARY_TABLE}
    WHERE {BASE_FILTER}
    GROUP BY Brand, VendorName
    """,
    conn,
)

sales_q1 = brand_perf["TotalSalesDollars"].quantile(0.25)
margin_q3 = brand_perf["ProfitMargin"].quantile(0.75)

adjustment_candidates = (
    brand_perf.query("TotalSalesDollars <= @sales_q1 and ProfitMargin >= @margin_q3")
    .sort_values(["ProfitMargin", "TotalSalesDollars"], ascending=[False, True])
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=brand_perf, x="TotalSalesDollars", y="ProfitMargin", alpha=0.2, s=25, color="gray")
sns.scatterplot(data=adjustment_candidates, x="TotalSalesDollars", y="ProfitMargin", s=90, color="crimson", edgecolor="black")
plt.axvline(sales_q1, ls="--", color="steelblue")
plt.axhline(margin_q3, ls="--", color="darkgreen")
plt.xscale("log")
plt.title("Low-Sales, High-Margin Brands")
plt.xlabel("Total Sales Dollars (log scale)")
plt.ylabel("Average Profit Margin")
plt.tight_layout()
plt.show()

print(f"Low-sales threshold (Q1): {sales_q1:,.2f}")
print(f"High-margin threshold (Q3): {margin_q3:,.2f}")
print(f"Priority brand-vendor pairs: {len(adjustment_candidates)}")
adjustment_candidates.head(25)

### Insights

- The saved thresholds identify low-sales brands as those below **$1,349.85** in sales and high-margin brands as those above **44.07%** average margin.
- Using those cutoffs, the notebook identified **25 priority brand-vendor pairs**.
- These brands are important because they are already profitable on a percentage basis but are not yet generating enough demand.
- Business interpretation: these are strong candidates for **promotion, better shelf placement, pricing experiments, or targeted sales enablement**, because increasing volume here may improve profit dollars without starting from weak economics.

## Question 6 — What is the relationship between purchase quantity and unit cost?

## Question 7 — What purchase quantity band appears optimal for cost savings?

In [ ]:
bulk_df = read_sql(
    """
    SELECT
        Quantity,
        PurchasePrice
    FROM purchases
    WHERE Quantity > 0 AND PurchasePrice > 0
    """,
    conn,
)

bulk_df["log_qty"] = np.log1p(bulk_df["Quantity"])
slope, intercept = np.polyfit(bulk_df["log_qty"], bulk_df["PurchasePrice"], 1)
price_drop_per_doubling = slope * np.log(2)

bulk_df["qty_band"] = pd.qcut(bulk_df["Quantity"], q=10, duplicates="drop")
band_summary = (
    bulk_df.groupby("qty_band", as_index=False)
    .agg(
        avg_qty=("Quantity", "mean"),
        min_qty=("Quantity", "min"),
        max_qty=("Quantity", "max"),
        avg_unit_price=("PurchasePrice", "mean"),
        line_count=("Quantity", "size"),
    )
    .sort_values("avg_qty")
)

min_lines = max(100, int(0.01 * len(bulk_df)))
stable_bands = band_summary[band_summary["line_count"] >= min_lines].copy()
optimal_band = stable_bands.loc[stable_bands["avg_unit_price"].idxmin()]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sample = bulk_df.sample(min(10000, len(bulk_df)), random_state=42)
axes[0].scatter(sample["Quantity"], sample["PurchasePrice"], alpha=0.15, s=10, color="slateblue")

x_grid = np.linspace(bulk_df["Quantity"].min(), bulk_df["Quantity"].quantile(0.99), 200)
y_grid = intercept + slope * np.log1p(x_grid)
axes[0].plot(x_grid, y_grid, color="crimson", linewidth=2)
axes[0].set_title("Unit Price vs Purchase Quantity")
axes[0].set_xlabel("Quantity")
axes[0].set_ylabel("Purchase Price")

axes[1].plot(band_summary["avg_qty"], band_summary["avg_unit_price"], marker="o", color="teal")
axes[1].axvspan(optimal_band["min_qty"], optimal_band["max_qty"], color="gold", alpha=0.25)
axes[1].set_title("Average Unit Price by Quantity Band")
axes[1].set_xlabel("Average Quantity in Band")
axes[1].set_ylabel("Average Unit Price")

plt.tight_layout()
plt.show()

pd.DataFrame(
    {
        "Metric": [
            "Rows analyzed",
            "Correlation (Quantity vs Unit Price)",
            "Estimated unit price change per quantity doubling",
            "Optimal quantity band for cost savings",
            "Average unit price in optimal band",
            "Band sample size",
        ],
        "Value": [
            f"{len(bulk_df):,}",
            f"{bulk_df['Quantity'].corr(bulk_df['PurchasePrice']):.3f}",
            f"{price_drop_per_doubling:.3f}",
            f"{int(optimal_band['min_qty'])} to {int(optimal_band['max_qty'])}",
            f"{optimal_band['avg_unit_price']:.2f}",
            f"{int(optimal_band['line_count']):,}",
        ],
    }
)

### Insights

- The saved run analyzed **2,372,321 purchase rows**.
- The correlation between quantity and unit price is **-0.083**, which indicates a **real but modest** negative relationship: bigger purchases are associated with lower prices, but the effect is not extremely strong overall.
- The log-fit estimate suggests unit price drops by about **2.260** for every quantity doubling.
- The most cost-efficient stable quantity band in the saved run was approximately **25 to 3,816 units**, with an average unit price of **8.44** across **190,517** lines.
- Business interpretation: buying more does appear to help, but the savings are not infinite. There seems to be a practical purchase band where cost efficiency improves before operational complexity or demand risk becomes too high.

## Question 8 — Which vendors have low inventory turnover?

## Question 9 — How much capital is locked in unsold inventory by vendor?

## Question 10 — Which vendors contribute the most to locked working capital?

In [ ]:
vendor_turnover = read_sql(
    f"""
    WITH vendor_turnover AS (
        SELECT
            VendorNumber,
            VendorName,
            SUM(TotalPurchaseQuantity) AS total_purchase_qty,
            SUM(TotalSalesQuantity) AS total_sales_qty,
            1.0 * SUM(TotalSalesQuantity) / NULLIF(SUM(TotalPurchaseQuantity), 0) AS inventory_turnover
        FROM {SUMMARY_TABLE}
        GROUP BY VendorNumber, VendorName
    )
    SELECT *
    FROM vendor_turnover
    WHERE inventory_turnover IS NOT NULL
    ORDER BY inventory_turnover ASC
    """,
    conn,
)

min_purchase_qty = vendor_turnover["total_purchase_qty"].quantile(0.25)
volume_filtered = vendor_turnover[vendor_turnover["total_purchase_qty"] >= min_purchase_qty].copy()
risk_threshold = volume_filtered["inventory_turnover"].quantile(0.25)
low_turnover_vendors = (
    volume_filtered[volume_filtered["inventory_turnover"] <= risk_threshold]
    .sort_values(["inventory_turnover", "total_purchase_qty"], ascending=[True, False])
    .reset_index(drop=True)
)

locked_capital_df = read_sql(
    f"""
    WITH vendor_brand_stock AS (
        SELECT
            VendorNumber,
            VendorName,
            Brand,
            SUM(TotalPurchaseQuantity) AS purchase_qty,
            SUM(TotalSalesQuantity) AS sales_qty,
            SUM(TotalPurchaseDollars) AS purchase_dollars
        FROM {SUMMARY_TABLE}
        GROUP BY VendorNumber, VendorName, Brand
    ), vendor_agg AS (
        SELECT
            VendorNumber,
            VendorName,
            SUM(CASE WHEN purchase_qty > sales_qty THEN purchase_qty - sales_qty ELSE 0 END) AS unsold_qty,
            SUM(CASE WHEN purchase_qty > 0 THEN purchase_dollars ELSE 0 END) AS purchase_dollars,
            SUM(CASE WHEN purchase_qty > 0 THEN purchase_qty ELSE 0 END) AS purchase_qty
        FROM vendor_brand_stock
        GROUP BY VendorNumber, VendorName
    )
    SELECT
        VendorNumber,
        VendorName,
        unsold_qty,
        CASE
            WHEN purchase_qty = 0 THEN 0
            ELSE 1.0 * purchase_dollars / purchase_qty
        END AS avg_unit_cost,
        unsold_qty * CASE
            WHEN purchase_qty = 0 THEN 0
            ELSE 1.0 * purchase_dollars / purchase_qty
        END AS locked_capital
    FROM vendor_agg
    WHERE unsold_qty > 0
    ORDER BY locked_capital DESC
    """,
    conn,
)

locked_capital_df["share_pct"] = locked_capital_df["locked_capital"] / locked_capital_df["locked_capital"].sum() * 100
locked_capital_df["cumulative_share_pct"] = locked_capital_df["share_pct"].cumsum()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.barplot(data=low_turnover_vendors.head(15), y="VendorName", x="inventory_turnover", color="indianred", ax=axes[0])
axes[0].axvline(risk_threshold, ls="--", color="black")
axes[0].set_title("Lowest Inventory Turnover Vendors")
axes[0].set_xlabel("Inventory Turnover")
axes[0].set_ylabel("Vendor")

sns.barplot(data=locked_capital_df.head(10), y="VendorName", x="locked_capital", color="firebrick", ax=axes[1])
axes[1].set_title("Top Vendors by Locked Working Capital")
axes[1].set_xlabel("Locked Capital")
axes[1].xaxis.set_major_formatter(FuncFormatter(fmt_millions))
add_bar_labels(axes[1], axis="x")

plt.tight_layout()
plt.show()

print(f"Low-turnover threshold (Q1): {risk_threshold:.3f}")
print(f"Vendors flagged after volume filter: {len(low_turnover_vendors)}")
print(f"Total locked capital: {locked_capital_df['locked_capital'].sum() / 1_000_000:.2f}M")
locked_capital_df.head(10)

### Insights

- The saved turnover screen used a low-turnover threshold of **0.927** after filtering out small-volume vendors.
- That process flagged **24 vendors** as slow-moving inventory risks.
- The saved locked-capital estimate shows approximately **$13.22M** tied up in unsold inventory across vendors.
- The largest single contributor to locked working capital was **MARTIGNETTI COMPANIES**, at **$1.39M**, representing **10.53%** of the total locked capital.
- Business interpretation: inventory risk is not only about low turnover; it is also a cash-flow issue. Vendors with high unsold value should be prioritized for replenishment review, assortment cleanup, and vendor negotiation.

## Question 11 — How does profit margin distribution differ between high- and low-performing vendors?

## Question 12 — What are the 95% confidence intervals for profit margin by performance group?

In [ ]:
perf_vendor = read_sql(
    f"""
    SELECT
        VendorNumber,
        VendorName,
        SUM(TotalSalesDollars) AS total_sales
    FROM {SUMMARY_TABLE}
    GROUP BY VendorNumber, VendorName
    HAVING SUM(TotalSalesDollars) > 0
    """,
    conn,
)

top_cutoff = perf_vendor["total_sales"].quantile(0.75)
low_cutoff = perf_vendor["total_sales"].quantile(0.25)

top_vendors = perf_vendor.loc[perf_vendor["total_sales"] >= top_cutoff, "VendorNumber"]
low_vendors = perf_vendor.loc[perf_vendor["total_sales"] <= low_cutoff, "VendorNumber"]

margin_df = read_sql(
    f"SELECT VendorNumber, ProfitMargin FROM {SUMMARY_TABLE} WHERE ProfitMargin IS NOT NULL",
    conn,
)

margin_compare = pd.concat(
    [
        margin_df[margin_df["VendorNumber"].isin(top_vendors)].assign(group="Top-performing vendors"),
        margin_df[margin_df["VendorNumber"].isin(low_vendors)].assign(group="Low-performing vendors"),
    ],
    ignore_index=True,
)

plt.figure(figsize=(10, 6))
sns.violinplot(data=margin_compare, x="group", y="ProfitMargin", inner="quartile", palette=["teal", "tomato"])
plt.title("Profit Margin Distribution by Vendor Performance Group")
plt.xlabel("Performance Group")
plt.ylabel("Profit Margin (%)")
plt.tight_layout()
plt.show()

summary_rows = []
for group_name, vendor_ids in {
    "Top-performing vendors": top_vendors,
    "Low-performing vendors": low_vendors,
}.items():
    group_margin = margin_df[margin_df["VendorNumber"].isin(vendor_ids)]["ProfitMargin"]
    mean, ci_low, ci_high = mean_ci(group_margin)
    summary_rows.append(
        {
            "Group": group_name,
            "Vendors in group": len(vendor_ids),
            "Observations": len(group_margin.dropna()),
            "Mean Profit Margin": mean,
            "95% CI Lower": ci_low,
            "95% CI Upper": ci_high,
        }
    )

ci_summary = pd.DataFrame(summary_rows)
ci_summary.style.format({
    "Mean Profit Margin": "{:.2f}%",
    "95% CI Lower": "{:.2f}%",
    "95% CI Upper": "{:.2f}%",
})

### Insights

- In the saved notebook output, **top-performing vendors** had an average profit margin of **-15.29%** with a **95% confidence interval from -25.12% to -5.45%**.
- **Low-performing vendors** had a weaker average profit margin of **-23.95%** with a much wider **95% confidence interval from -54.22% to 6.31%**.
- The wider interval for low-performing vendors suggests their margin behavior is much more volatile and less predictable.
- Even the stronger group still shows a negative mean margin in the saved output, which is a sign that the underlying pricing/profit logic should be reviewed carefully in the next data refresh. That could reflect promotional intensity, freight/tax treatment, cost allocation rules, or a business model nuance in the source data.
- Business interpretation: higher-sales vendors appear materially healthier than lower-sales vendors, but the entire margin structure deserves a second validation pass because sustained negative mean margins are operationally important.

In [ ]:
conn.close()